## Silver Layer Transformation

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    LongType, DoubleType, StringType, TimestampType, BooleanType, DecimalType
)
import time

CATALOG = "ecommerce"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SILVER_SCHEMA}")

print(f"Catalog: {CATALOG}")
print(f"Source:  {BRONZE_SCHEMA}")
print(f"Target:  {SILVER_SCHEMA}")

Catalog: ecommerce
Source:  bronze
Target:  silver


#### Transform: Customers

In [0]:
def transform_customers():
    df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.users_raw")

    df_clean = (
        df
        # Rename PK
        .withColumnRenamed("id", "user_id")
        # Fix string "null" → SQL NULL (913 rows in city)
        .withColumn("city", F.when(F.col("city") == "null", F.lit(None)).otherwise(F.col("city")))
        # Standardize country
        .withColumn("country", F.when(F.col("country") == "Brasil", "Brazil").otherwise(F.col("country")))
        # Enforce schema types explicitly
        .withColumn("user_id", F.col("user_id").cast(LongType()))
        .withColumn("first_name", F.col("first_name").cast(StringType()))
        .withColumn("last_name", F.col("last_name").cast(StringType()))
        .withColumn("email", F.col("email").cast(StringType()))
        .withColumn("age", F.col("age").cast(LongType()))
        .withColumn("gender", F.col("gender").cast(StringType()))
        .withColumn("state", F.col("state").cast(StringType()))
        .withColumn("street_address", F.col("street_address").cast(StringType()))
        .withColumn("postal_code", F.col("postal_code").cast(StringType()))
        .withColumn("city", F.col("city").cast(StringType()))
        .withColumn("country", F.col("country").cast(StringType()))
        .withColumn("latitude", F.col("latitude").cast(DoubleType()))
        .withColumn("longitude", F.col("longitude").cast(DoubleType()))
        .withColumn("traffic_source", F.col("traffic_source").cast(StringType()))
        .withColumn("created_at", F.col("created_at").cast(TimestampType()))
        
        .select(
            "user_id", "first_name", "last_name", "email", "age", "gender",
            "state", "street_address", "postal_code", "city", "country",
            "latitude", "longitude", "traffic_source", "created_at"
        )
    )

    return df_clean

#### Transform: Orders

In [0]:
def transform_orders():
    df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.orders_raw")

    df_clean = (
        df
        # Enforce schema types
        .withColumn("order_id", F.col("order_id").cast(LongType()))
        .withColumn("user_id", F.col("user_id").cast(LongType()))
        .withColumn("status", F.col("status").cast(StringType()))
        .withColumn("created_at", F.col("created_at").cast(TimestampType()))
        .withColumn("returned_at", F.col("returned_at").cast(TimestampType()))
        .withColumn("shipped_at", F.col("shipped_at").cast(TimestampType()))
        .withColumn("delivered_at", F.col("delivered_at").cast(TimestampType()))
        .withColumn("num_of_item", F.col("num_of_item").cast(LongType()))
        # Derived columns
        .withColumn(
            "days_to_ship",
            F.datediff(F.col("shipped_at"), F.col("created_at"))
        )
        .withColumn(
            "days_to_deliver",
            F.datediff(F.col("delivered_at"), F.col("created_at"))
        )
       
        .select(
            "order_id", "user_id", "status", "created_at",
            "shipped_at", "delivered_at", "returned_at",
            "num_of_item", "days_to_ship", "days_to_deliver"
        )
    )

    return df_clean

#### Transform: Order Items

In [0]:
def transform_order_items():
    df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.order_items_raw")

    df_clean = (
        df
        # Rename PK
        .withColumnRenamed("id", "order_item_id")
        # Enforce schema types
        .withColumn("order_item_id", F.col("order_item_id").cast(LongType()))
        .withColumn("order_id", F.col("order_id").cast(LongType()))
        .withColumn("user_id", F.col("user_id").cast(LongType()))
        .withColumn("product_id", F.col("product_id").cast(LongType()))
        .withColumn("inventory_item_id", F.col("inventory_item_id").cast(LongType()))
        .withColumn("status", F.col("status").cast(StringType()))
        .withColumn("created_at", F.col("created_at").cast(TimestampType()))
        .withColumn("shipped_at", F.col("shipped_at").cast(TimestampType()))
        .withColumn("delivered_at", F.col("delivered_at").cast(TimestampType()))
        .withColumn("returned_at", F.col("returned_at").cast(TimestampType()))
        .withColumn("sale_price", F.col("sale_price").cast(DoubleType()))
        # Derived columns
        .withColumn("is_returned", (F.col("status") == "Returned").cast(BooleanType()))
        
        .select(
            "order_item_id", "order_id", "user_id", "product_id",
            "inventory_item_id", "status", "created_at", "shipped_at",
            "delivered_at", "returned_at", "sale_price", "is_returned"
        )
    )

    return df_clean

#### Transform: Products

In [0]:
def transform_products():
    df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.products_raw")

    df_clean = (
        df
        # Rename PK
        .withColumnRenamed("id", "product_id")
        # Enforce schema types
        .withColumn("product_id", F.col("product_id").cast(LongType()))
        .withColumn("cost", F.col("cost").cast(DoubleType()))
        .withColumn("category", F.col("category").cast(StringType()))
        .withColumn("name", F.col("name").cast(StringType()))
        .withColumn("brand", F.col("brand").cast(StringType()))
        .withColumn("retail_price", F.col("retail_price").cast(DoubleType()))
        .withColumn("department", F.col("department").cast(StringType()))
        .withColumn("sku", F.col("sku").cast(StringType()))
        .withColumn("distribution_center_id", F.col("distribution_center_id").cast(LongType()))
        # Derived columns
        .withColumn("base_margin", F.round(F.col("retail_price") - F.col("cost"), 2))
        
        .select(
            "product_id", "cost", "category", "name", "brand",
            "retail_price", "department", "sku", "distribution_center_id",
            "base_margin"
        )
    )

    return df_clean

#### Transform: Inventory

In [0]:
def transform_inventory():
    df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.inventory_items_raw")

    df_clean = (
        df
        # Rename PK
        .withColumnRenamed("id", "inventory_item_id")
        # Enforce schema types
        .withColumn("inventory_item_id", F.col("inventory_item_id").cast(LongType()))
        .withColumn("product_id", F.col("product_id").cast(LongType()))
        .withColumn("created_at", F.col("created_at").cast(TimestampType()))
        .withColumn("sold_at", F.col("sold_at").cast(TimestampType()))
        .withColumn("cost", F.col("cost").cast(DoubleType()))
        # Derived columns
        .withColumn("is_sold", F.col("sold_at").isNotNull().cast(BooleanType()))
        .withColumn(
            "days_to_sell",
            F.datediff(F.col("sold_at"), F.col("created_at"))
        )
        
        .select(
            "inventory_item_id", "product_id", "created_at", "sold_at",
            "cost", "is_sold", "days_to_sell"
        )
    )

    return df_clean

#### Transform: Events

In [0]:
def transform_events():
    df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.events_raw")

    df_clean = (
        df
        # Rename PK
        .withColumnRenamed("id", "event_id")
        # Enforce schema types
        .withColumn("event_id", F.col("event_id").cast(LongType()))
        .withColumn("user_id", F.col("user_id").cast(LongType()))
        .withColumn("sequence_number", F.col("sequence_number").cast(LongType()))
        .withColumn("session_id", F.col("session_id").cast(StringType()))
        .withColumn("created_at", F.col("created_at").cast(TimestampType()))
        .withColumn("city", F.col("city").cast(StringType()))
        .withColumn("state", F.col("state").cast(StringType()))
        .withColumn("postal_code", F.col("postal_code").cast(StringType()))
        .withColumn("browser", F.col("browser").cast(StringType()))
        .withColumn("traffic_source", F.col("traffic_source").cast(StringType()))
        .withColumn("uri", F.col("uri").cast(StringType()))
        .withColumn("event_type", F.col("event_type").cast(StringType()))
        # Derived columns
        .withColumn("is_anonymous", F.col("user_id").isNull().cast(BooleanType()))
        
        .select(
            "event_id", "user_id", "sequence_number", "session_id",
            "created_at", "city", "state", "postal_code", "browser",
            "traffic_source", "uri", "event_type", "is_anonymous"
        )
    )

    return df_clean

#### Transform: Distribution Centers

In [0]:
def transform_distribution_centers():
    df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.distribution_centers_raw")

    df_clean = (
        df
        # Rename PK
        .withColumnRenamed("id", "distribution_center_id")
        # Enforce schema types
        .withColumn("distribution_center_id", F.col("distribution_center_id").cast(LongType()))
        .withColumn("name", F.col("name").cast(StringType()))
        .withColumn("latitude", F.col("latitude").cast(DoubleType()))
        .withColumn("longitude", F.col("longitude").cast(DoubleType()))
        
        .select(
            "distribution_center_id", "name", "latitude", "longitude"
        )
    )

    return df_clean

#### Executing All Silver Transformations

In [0]:
TRANSFORMS = {
    "customers_clean":             transform_customers,
    "orders_clean":                transform_orders,
    "order_items_clean":           transform_order_items,
    "products_clean":              transform_products,
    "inventory_clean":             transform_inventory,
    "events_clean":                transform_events,
    "distribution_centers_clean":  transform_distribution_centers,
}

results = []
total_start = time.time()

for table_name, transform_fn in TRANSFORMS.items():
    t0 = time.time()
    full_table = f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"

    try:
        df = transform_fn()
        row_count = df.count()

        # Write as managed Delta table (overwrite for idempotency)
        df.write.format("delta").mode("overwrite").saveAsTable(full_table)

        duration = round(time.time() - t0, 1)
        results.append({
            "table": table_name,
            "rows": row_count,
            "columns": len(df.columns),
            "duration_s": duration,
            "status": "✓"
        })
        print(f"  ✓ {table_name:<35} {row_count:>10,} rows | {len(df.columns):>3} cols | {duration:>5.1f}s")

    except Exception as e:
        duration = round(time.time() - t0, 1)
        results.append({
            "table": table_name,
            "rows": 0,
            "columns": 0,
            "duration_s": duration,
            "status": "✗"
        })
        print(f"  ✗ {table_name:<35} FAILED after {duration:.1f}s — {str(e)[:120]}")

total_duration = round(time.time() - total_start, 1)
total_rows = sum(r["rows"] for r in results)
success_count = sum(1 for r in results if r["status"] == "✓")

print(f"\n{'='*70}")
print(f"  {success_count}/{len(TRANSFORMS)} tables written | {total_rows:,} total rows | {total_duration:.1f}s")
print(f"{'='*70}")

  ✓ customers_clean                        100,000 rows |  15 cols |  21.1s
  ✓ orders_clean                           124,850 rows |  10 cols |   6.2s
  ✓ order_items_clean                      181,424 rows |  12 cols |   6.5s
  ✓ products_clean                          29,120 rows |  10 cols |   5.6s
  ✓ inventory_clean                        490,083 rows |   7 cols |   5.8s
  ✓ events_clean                         2,427,889 rows |  13 cols |   8.1s
  ✓ distribution_centers_clean                  10 rows |   4 cols |   5.2s

  7/7 tables written | 3,353,376 total rows | 60.2s


#### Delta Check

In [0]:
CONSTRAINTS = {
    "customers_clean": [
        ("pk_not_null", "user_id IS NOT NULL"),
    ],
    "orders_clean": [
        ("pk_not_null", "order_id IS NOT NULL"),
    ],
    "order_items_clean": [
        ("pk_not_null", "order_item_id IS NOT NULL"),
        ("positive_price", "sale_price > 0"),
    ],
    "products_clean": [
        ("pk_not_null", "product_id IS NOT NULL"),
    ],
    "inventory_clean": [
        ("pk_not_null", "inventory_item_id IS NOT NULL"),
    ],
    "events_clean": [
        ("pk_not_null", "event_id IS NOT NULL"),
    ],
    "distribution_centers_clean": [
        ("pk_not_null", "distribution_center_id IS NOT NULL"),
    ],
}

print("Applying Delta CHECK constraints...\n")

for table_name, constraints in CONSTRAINTS.items():
    full_table = f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    for constraint_name, expression in constraints:
        try:
            spark.sql(f"ALTER TABLE {full_table} ADD CONSTRAINT {constraint_name} CHECK ({expression})")
            print(f"  ✓ {table_name}.{constraint_name}: {expression}")
        except Exception as e:
            error_msg = str(e)
            if "CONSTRAINT_ALREADY_EXISTS" in error_msg or "already exists" in error_msg.lower():
                print(f"  {table_name}.{constraint_name}: already exists (idempotent)")
            else:
                print(f"  {table_name}.{constraint_name}: FAILED — {error_msg[:100]}")

print("\nConstraints applied.")

Applying Delta CHECK constraints...

  ✓ customers_clean.pk_not_null: user_id IS NOT NULL
  ✓ orders_clean.pk_not_null: order_id IS NOT NULL
  ✓ order_items_clean.pk_not_null: order_item_id IS NOT NULL
  ✓ order_items_clean.positive_price: sale_price > 0
  ✓ products_clean.pk_not_null: product_id IS NOT NULL
  ✓ inventory_clean.pk_not_null: inventory_item_id IS NOT NULL
  ✓ events_clean.pk_not_null: event_id IS NOT NULL
  ✓ distribution_centers_clean.pk_not_null: distribution_center_id IS NOT NULL

Constraints applied.


#### Post Transformation Validation

In [0]:
print("=" * 70)
print("  SILVER LAYER VALIDATION")
print("=" * 70)

# Expected row counts from profiling
EXPECTED_COUNTS = {
    "customers_clean":             100_000,
    "orders_clean":                124_850,
    "order_items_clean":           181_424,
    "products_clean":              29_120,
    "inventory_clean":             490_083,
    "events_clean":                2_427_889,
    "distribution_centers_clean":  10,
}

print(f"\n{'Table':<35} {'Expected':>10} {'Actual':>10} {'Status':>8}")
print("-" * 70)

all_pass = True
for table_name, expected in EXPECTED_COUNTS.items():
    full_table = f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    actual = spark.table(full_table).count()
    status = "PASS" if actual == expected else "FAIL"
    if actual != expected:
        all_pass = False
    print(f"  {table_name:<33} {expected:>10,} {actual:>10,} {status:>8}")

print("-" * 70)
if all_pass:
    print("  All row counts match. Silver layer is consistent with Bronze.")
else:
    print("  Row count mismatch detected. Investigate before proceeding to Gold.")

  SILVER LAYER VALIDATION

Table                                 Expected     Actual   Status
----------------------------------------------------------------------
  customers_clean                      100,000    100,000     PASS
  orders_clean                         124,850    124,850     PASS
  order_items_clean                    181,424    181,424     PASS
  products_clean                        29,120     29,120     PASS
  inventory_clean                      490,083    490,083     PASS
  events_clean                       2,427,889  2,427,889     PASS
  distribution_centers_clean                10         10     PASS
----------------------------------------------------------------------
  All row counts match. Silver layer is consistent with Bronze.


#### Verify Derived Columns

In [0]:
print("\n" + "=" * 70)
print("  SILVER DERIVED COLUMN SPOT CHECKS")
print("=" * 70)

# 1. Verify string "null" 
customers = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.customers_clean")
string_nulls = customers.filter(F.col("city") == "null").count()
sql_nulls = customers.filter(F.col("city").isNull()).count()
print(f"\n  customers_clean.city:")
print(f"    String 'null' remaining: {string_nulls} (should be 0)")
print(f"    SQL NULL count: {sql_nulls} (should be >= 913)")

# 2. Verify Brasil → Brazil
brasil_count = customers.filter(F.col("country") == "Brasil").count()
brazil_count = customers.filter(F.col("country") == "Brazil").count()
print(f"\n  customers_clean.country:")
print(f"    'Brasil' remaining: {brasil_count} (should be 0)")
print(f"    'Brazil' count: {brazil_count}")

# 3. Verify orders derived columns
orders = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.orders_clean")
shipped_orders = orders.filter(F.col("shipped_at").isNotNull())
has_days_to_ship = shipped_orders.filter(F.col("days_to_ship").isNotNull()).count()
print(f"\n  orders_clean.days_to_ship:")
print(f"    Shipped orders: {shipped_orders.count():,}")
print(f"    With days_to_ship: {has_days_to_ship:,} (should match)")

# 4. Verify order_items is_returned flag
items = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.order_items_clean")
returned_by_status = items.filter(F.col("status") == "Returned").count()
returned_by_flag = items.filter(F.col("is_returned") == True).count()
print(f"\n  order_items_clean.is_returned:")
print(f"    status='Returned': {returned_by_status:,}")
print(f"    is_returned=True:  {returned_by_flag:,} (should match)")

# 5. Verify inventory denormalized columns dropped
inventory = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.inventory_clean")
inv_cols = inventory.columns
print(f"\n  inventory_clean columns ({len(inv_cols)}): {inv_cols}")
has_denormalized = any(c.startswith("product_") and c != "product_id" for c in inv_cols)
print(f"    Denormalized product_* columns present: {has_denormalized} (should be False)")

# 6. Verify events PII dropped
events = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.events_clean")
evt_cols = events.columns
print(f"\n  events_clean columns ({len(evt_cols)}): {evt_cols}")
has_ip = "ip_address" in evt_cols
print(f"    ip_address present: {has_ip} (should be False)")

# 7. Verify products — NULL brands/names preserved
products = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.products_clean")
null_brands = products.filter(F.col("brand").isNull()).count()
unknown_brands = products.filter(F.col("brand") == "Unknown").count()
print(f"\n  products_clean.brand:")
print(f"    NULL brands: {null_brands} (should be 24)")
print(f"    'Unknown' brands: {unknown_brands} (should be 0)")

print(f"\n{'='*70}")
print("  Spot checks complete.")
print(f"{'='*70}")


  SILVER DERIVED COLUMN SPOT CHECKS

  customers_clean.city:
    String 'null' remaining: 0 (should be 0)
    SQL NULL count: 913 (should be >= 913)

  customers_clean.country:
    'Brasil' remaining: 0 (should be 0)
    'Brazil' count: 14551

  orders_clean.days_to_ship:
    Shipped orders: 80,902
    With days_to_ship: 80,902 (should match)

  order_items_clean.is_returned:
    status='Returned': 17,914
    is_returned=True:  17,914 (should match)

  inventory_clean columns (7): ['inventory_item_id', 'product_id', 'created_at', 'sold_at', 'cost', 'is_sold', 'days_to_sell']
    Denormalized product_* columns present: False (should be False)

  events_clean columns (13): ['event_id', 'user_id', 'sequence_number', 'session_id', 'created_at', 'city', 'state', 'postal_code', 'browser', 'traffic_source', 'uri', 'event_type', 'is_anonymous']
    ip_address present: False (should be False)

  products_clean.brand:
    NULL brands: 24 (should be 24)
    'Unknown' brands: 0 (should be 0)

  S

#### Additional Delta Checks

In [0]:
ADDITIONAL_CONSTRAINTS = {
    "orders_clean": [
        ("valid_days_to_ship", "days_to_ship >= 0 OR days_to_ship IS NULL"),
        ("valid_days_to_deliver", "days_to_deliver >= 0 OR days_to_deliver IS NULL"),
    ],
    "order_items_clean": [
        ("positive_sale_price", "sale_price > 0"),
    ],
    "products_clean": [
        ("valid_margin", "retail_price >= cost"),
    ],
    "inventory_clean": [
        ("valid_days_to_sell", "days_to_sell >= 0 OR days_to_sell IS NULL"),
    ],
}

print("Applying additional business rule constraints...\n")

for table_name, constraints in ADDITIONAL_CONSTRAINTS.items():
    full_table = f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    for constraint_name, expression in constraints:
        try:
            spark.sql(f"ALTER TABLE {full_table} ADD CONSTRAINT {constraint_name} CHECK ({expression})")
            print(f"  ✓ {table_name}.{constraint_name}: {expression}")
        except Exception as e:
            error_msg = str(e)
            if "CONSTRAINT_ALREADY_EXISTS" in error_msg or "already exists" in error_msg.lower():
                print(f"  {table_name}.{constraint_name}: already exists (idempotent)")
            else:
                print(f"  {table_name}.{constraint_name}: FAILED — {error_msg[:120]}")

print("\nBusiness rule constraints applied.")

Applying additional business rule constraints...

  ✓ orders_clean.valid_days_to_ship: days_to_ship >= 0 OR days_to_ship IS NULL
  ✓ orders_clean.valid_days_to_deliver: days_to_deliver >= 0 OR days_to_deliver IS NULL
  ✓ order_items_clean.positive_sale_price: sale_price > 0
  ✓ products_clean.valid_margin: retail_price >= cost
  ✓ inventory_clean.valid_days_to_sell: days_to_sell >= 0 OR days_to_sell IS NULL

Business rule constraints applied.


#### Foreign Key Validation

In [0]:
print("=" * 70)
print("  FOREIGN KEY VALIDATION")
print("=" * 70)

FK_CHECKS = [
    {
        "name": "orders → customers",
        "child_table": f"{CATALOG}.{SILVER_SCHEMA}.orders_clean",
        "child_key": "user_id",
        "parent_table": f"{CATALOG}.{SILVER_SCHEMA}.customers_clean",
        "parent_key": "user_id",
    },
    {
        "name": "order_items → orders",
        "child_table": f"{CATALOG}.{SILVER_SCHEMA}.order_items_clean",
        "child_key": "order_id",
        "parent_table": f"{CATALOG}.{SILVER_SCHEMA}.orders_clean",
        "parent_key": "order_id",
    },
    {
        "name": "order_items → customers",
        "child_table": f"{CATALOG}.{SILVER_SCHEMA}.order_items_clean",
        "child_key": "user_id",
        "parent_table": f"{CATALOG}.{SILVER_SCHEMA}.customers_clean",
        "parent_key": "user_id",
    },
    {
        "name": "order_items → products",
        "child_table": f"{CATALOG}.{SILVER_SCHEMA}.order_items_clean",
        "child_key": "product_id",
        "parent_table": f"{CATALOG}.{SILVER_SCHEMA}.products_clean",
        "parent_key": "product_id",
    },
    {
        "name": "inventory → products",
        "child_table": f"{CATALOG}.{SILVER_SCHEMA}.inventory_clean",
        "child_key": "product_id",
        "parent_table": f"{CATALOG}.{SILVER_SCHEMA}.products_clean",
        "parent_key": "product_id",
    },
    {
        "name": "products → distribution_centers",
        "child_table": f"{CATALOG}.{SILVER_SCHEMA}.products_clean",
        "child_key": "distribution_center_id",
        "parent_table": f"{CATALOG}.{SILVER_SCHEMA}.distribution_centers_clean",
        "parent_key": "distribution_center_id",
    },
]

print(f"\n{'Relationship':<35} {'Orphans':>10} {'Status':>8}")
print("-" * 58)

all_fk_pass = True
for check in FK_CHECKS:
    child_df = spark.table(check["child_table"])
    parent_df = spark.table(check["parent_table"])
    
    # Count child keys that don't exist in parent
    orphan_count = (
        child_df
        .select(check["child_key"])
        .filter(F.col(check["child_key"]).isNotNull())
        .join(
            parent_df.select(check["parent_key"]),
            child_df[check["child_key"]] == parent_df[check["parent_key"]],
            "left_anti"
        )
        .count()
    )
    
    status = "PASS" if orphan_count == 0 else "FAIL"
    if orphan_count > 0:
        all_fk_pass = False
    print(f"  {check['name']:<33} {orphan_count:>10,} {status:>8}")

print("-" * 58)
if all_fk_pass:
    print("All foreign key relationships validated. Zero orphans.")
else:
    print("Orphan records found. Investigate before proceeding to Gold.")

  FOREIGN KEY VALIDATION

Relationship                           Orphans   Status
----------------------------------------------------------
  orders → customers                         0     PASS
  order_items → orders                       0     PASS
  order_items → customers                    0     PASS
  order_items → products                     0     PASS
  inventory → products                       0     PASS
  products → distribution_centers            0     PASS
----------------------------------------------------------
All foreign key relationships validated. Zero orphans.


In [0]:
display(spark.table(f"{CATALOG}.{SILVER_SCHEMA}.customers_clean").limit(5))

user_id,first_name,last_name,email,age,gender,state,street_address,postal_code,city,country,latitude,longitude,traffic_source,created_at
60914,Jessica,Watson,jessicawatson@example.net,43,F,Acre,7165 Hall Squares Apt. 968,69980-000,null,Brazil,-8.065346116,-72.87094866,Facebook,2025-09-12T07:24:00.000Z
4688,Sabrina,Jones,sabrinajones@example.net,15,F,Acre,6098 Timothy Stream Apt. 682,69980-000,null,Brazil,-8.065346116,-72.87094866,Search,2023-05-01T10:06:00.000Z
91933,Paula,Peterson,paulapeterson@example.net,31,F,Acre,8637 Martin Coves,69980-000,null,Brazil,-8.065346116,-72.87094866,Organic,2019-11-10T05:30:00.000Z
12362,James,Adams,jamesadams@example.com,20,M,Acre,343 Ryan Pines Suite 714,69980-000,null,Brazil,-8.065346116,-72.87094866,Organic,2024-05-13T17:22:00.000Z
76571,Nicole,Jones,nicolejones@example.org,67,F,Acre,25055 Lee Ridge Suite 348,69980-000,null,Brazil,-8.065346116,-72.87094866,Organic,2021-02-12T15:53:00.000Z
